In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!nvidia-smi

Sat May 16 18:19:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q python-docx pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.1 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
import pandas as pd
import re

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

sentences = []

for item in dataset:
    text = item["text"].strip().lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text.split()) >= 5:
        sentences.append(text)

df = pd.DataFrame({"Text": sentences})
df.to_csv("output.csv", index=False)

print("Total sentences:", len(df))
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Total sentences: 16932


,Text
0,senj no valkyria unrecorded chronicles japanes...
1,the game began development in carrying over a ...
2,it met with positive sales in japan and was pr...
3,as with previous valkyira chronicles games val...
4,the game s battle system the blitz system is c...


In [5]:
import os
import re
import string
import pandas as pd

if "output.csv" in os.listdir():
    df = pd.read_csv("output.csv")
    df = df.dropna()
    df["Text"] = df["Text"].astype(str).str.lower()
    print("Using existing output.csv")

else:
    from docx import Document

    doc_path = "wikipedia.docx"
    doc = Document(doc_path)

    text_data = [paragraph.text for paragraph in doc.paragraphs]
    text_data = [text.lower() for text in text_data]
    text_data = [re.sub(r"\[.*?\]", "", text) for text in text_data]

    english_alphabet = set(string.ascii_lowercase)

    cleaned_data = []

    for text in text_data:
        words = text.split()
        clean_words = []

        for word in words:
            word = re.sub(r"[^a-z]", "", word)
            if word and all(char in english_alphabet for char in word):
                clean_words.append(word)

        sentence = " ".join(clean_words).strip()

        if sentence:
            cleaned_data.append(sentence)

    df = pd.DataFrame({"Text": cleaned_data})
    df.to_csv("output.csv", index=False)

    print("Created output.csv")

df.head()


Using existing output.csv


,Text
0,senj no valkyria unrecorded chronicles japanes...
1,the game began development in carrying over a ...
2,it met with positive sales in japan and was pr...
3,as with previous valkyira chronicles games val...
4,the game s battle system the blitz system is c...


In [6]:
from collections import Counter
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, csv_path, sequence_length=5, max_vocab_size=10000):
        self.sequence_length = sequence_length

        df = pd.read_csv(csv_path)
        text = df["Text"].dropna().astype(str).str.cat(sep=" ").lower()

        words = re.findall(r"[a-z]+", text)

        word_counts = Counter(words)
        most_common = word_counts.most_common(max_vocab_size - 1)

        self.unique_words = ["<UNK>"] + [word for word, count in most_common]

        self.word_to_index = {
            word: index for index, word in enumerate(self.unique_words)
        }

        self.index_to_word = {
            index: word for word, index in self.word_to_index.items()
        }

        self.word_indexes = [
            self.word_to_index.get(word, self.word_to_index["<UNK>"])
            for word in words
        ]

    def __len__(self):
        return len(self.word_indexes) - self.sequence_length

    def __getitem__(self, index):
        x = self.word_indexes[index:index + self.sequence_length]
        y = self.word_indexes[index + 1:index + self.sequence_length + 1]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


In [7]:
from torch import nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, lstm_size=128, num_layers=2):
        super().__init__()

        self.lstm_size = lstm_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_size,
            num_layers=num_layers,
            dropout=0.2,
            batch_first=True
        )

        self.fc = nn.Linear(lstm_size, vocab_size)

    def forward(self, x, hidden=None):
        embed = self.embedding(x)
        output, hidden = self.lstm(embed, hidden)
        logits = self.fc(output)

        return logits, hidden


In [8]:
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sequence_length = 5
batch_size = 64
learning_rate = 0.001
num_epochs = 10
max_vocab_size = 10000

dataset = TextDataset(
    csv_path="output.csv",
    sequence_length=sequence_length,
    max_vocab_size=max_vocab_size
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = LSTMModel(vocab_size=len(dataset.unique_words)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs, _ = model(inputs)

        loss = criterion(
            outputs.reshape(-1, len(dataset.unique_words)),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs, _ = model(inputs)

            loss = criterion(
                outputs.reshape(-1, len(dataset.unique_words)),
                targets.reshape(-1)
            )

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f}"
    )


Device: cuda
Epoch [1/10] Train Loss: 5.6491 Val Loss: 5.2909
Epoch [2/10] Train Loss: 5.2075 Val Loss: 5.0906
Epoch [3/10] Train Loss: 5.0607 Val Loss: 4.9957
Epoch [4/10] Train Loss: 4.9767 Val Loss: 4.9350
Epoch [5/10] Train Loss: 4.9203 Val Loss: 4.8925
Epoch [6/10] Train Loss: 4.8792 Val Loss: 4.8613
Epoch [7/10] Train Loss: 4.8467 Val Loss: 4.8375
Epoch [8/10] Train Loss: 4.8216 Val Loss: 4.8189
Epoch [9/10] Train Loss: 4.8004 Val Loss: 4.8011
Epoch [10/10] Train Loss: 4.7831 Val Loss: 4.7857


In [9]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "word_to_index": dataset.word_to_index,
    "index_to_word": dataset.index_to_word,
    "unique_words": dataset.unique_words,
    "sequence_length": dataset.sequence_length,
    "vocab_size": len(dataset.unique_words),
    "embedding_dim": 128,
    "lstm_size": 128,
    "num_layers": 2
}

torch.save(checkpoint, "autocomplete_lstm_epoch10.pth")

print("Saved model after 10 epochs")

Saved model after 10 epochs


In [10]:
extra_epochs = 5
start_epoch = 10

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(extra_epochs):
    model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs, _ = model(inputs)

        loss = criterion(
            outputs.reshape(-1, len(dataset.unique_words)),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs, _ = model(inputs)

            loss = criterion(
                outputs.reshape(-1, len(dataset.unique_words)),
                targets.reshape(-1)
            )

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch [{start_epoch + epoch + 1}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f}"
    )

Epoch [11] Train Loss: 4.7350 Val Loss: 4.7478
Epoch [12] Train Loss: 4.7091 Val Loss: 4.7367
Epoch [13] Train Loss: 4.6977 Val Loss: 4.7290
Epoch [14] Train Loss: 4.6887 Val Loss: 4.7212
Epoch [15] Train Loss: 4.6812 Val Loss: 4.7148


In [12]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "word_to_index": dataset.word_to_index,
    "index_to_word": dataset.index_to_word,
    "unique_words": dataset.unique_words,
    "sequence_length": dataset.sequence_length,
    "vocab_size": len(dataset.unique_words),
    "embedding_dim": 128,
    "lstm_size": 128,
    "num_layers": 2
}

torch.save(checkpoint, "autocomplete_lstm_epoch15.pth")

print("Saved model after 15 epochs")

Saved model after 15 epochs


In [13]:
extra_epochs = 5
start_epoch = 15

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(extra_epochs):
    model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs, _ = model(inputs)

        loss = criterion(
            outputs.reshape(-1, len(dataset.unique_words)),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs, _ = model(inputs)

            loss = criterion(
                outputs.reshape(-1, len(dataset.unique_words)),
                targets.reshape(-1)
            )

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch [{start_epoch + epoch + 1}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f}"
    )

Epoch [16] Train Loss: 4.7010 Val Loss: 4.7209
Epoch [17] Train Loss: 4.6806 Val Loss: 4.7122
Epoch [18] Train Loss: 4.6716 Val Loss: 4.7059
Epoch [19] Train Loss: 4.6644 Val Loss: 4.7002
Epoch [20] Train Loss: 4.6583 Val Loss: 4.6949


In [14]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "word_to_index": dataset.word_to_index,
    "index_to_word": dataset.index_to_word,
    "unique_words": dataset.unique_words,
    "sequence_length": dataset.sequence_length,
    "vocab_size": len(dataset.unique_words),
    "embedding_dim": 128,
    "lstm_size": 128,
    "num_layers": 2
}

torch.save(checkpoint, "autocomplete_lstm.pth")

print("Saved final model")

Saved final model


In [15]:
import torch.nn.functional as F

def predict_next_words(model, text, dataset, top_k=5):
    model.eval()

    words = re.findall(r"[a-z]+", text.lower())

    if not words:
        return []

    words = words[-dataset.sequence_length:]

    indexes = [
        dataset.word_to_index.get(word, dataset.word_to_index["<UNK>"])
        for word in words
    ]

    input_tensor = torch.tensor([indexes], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs, _ = model(input_tensor)
        last_logits = outputs[0, -1]
        probabilities = F.softmax(last_logits, dim=0)

        top_indexes = torch.topk(probabilities, top_k).indices.tolist()

    suggestions = [
        dataset.index_to_word[index]
        for index in top_indexes
        if dataset.index_to_word[index] != "<UNK>"
    ]

    return suggestions


test_sentence = "artificial intelligence is"
print(predict_next_words(model, test_sentence, dataset))


['a', 'the', 'to', 'not']


In [16]:
from google.colab import files

files.download("autocomplete_lstm.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>